# Carga de datos · Credit Payment MLOps

## Objetivo
Leer la base entregada, verificar su estructura y exportarla a CSV sin alterar el Excel.

**Fuente:** `Base_de_datos.xlsx`, hoja `Hoja1`. Cada fila se trata como un crédito; no hay identificador de cliente para verificar si una persona aparece varias veces.

**Supuesto pendiente:** `Pago_atiempo=1` significa pago a tiempo, y `0` fuera de plazo. La interpretación se deduce del nombre; no se recibió un diccionario oficial.

## 1. Importaciones y rutas

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
ROOT = Path.cwd().resolve()
if not (ROOT / "Base_de_datos.xlsx").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.fe_engineering import load_data, temporal_split, CreditFeatures, FEATURES
sns.set_theme(style="whitegrid", palette=["#245A81", "#C47735"])
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.titlesize": 13})

## 2. Leer la fuente original

In [2]:
source = ROOT / "Base_de_datos.xlsx"
df = load_data(source)
print(f"Filas: {len(df):,} | Columnas: {df.shape[1]}")
display(df.head(5))

Filas: 10,763 | Columnas: 23


,tipo_credito,fecha_prestamo,capital_prestado,plazo_meses,edad_cliente,tipo_laboral,salario_cliente,total_otros_prestamos,cuota_pactada,puntaje,...,saldo_mora,saldo_total,saldo_principal,saldo_mora_codeudor,creditos_sectorFinanciero,creditos_sectorCooperativo,creditos_sectorReal,promedio_ingresos_datacredito,tendencia_ingresos,Pago_atiempo
0,7,2024-12-21 11:31:35,3692160.0,10,42,Independiente,8000000,2500000,341296,88.768094,...,0.0,51258.0,51258.0,0.0,5,0,0,908526.0,Estable,1
1,4,2025-04-22 09:47:35,840000.0,6,60,Empleado,3000000,2000000,124876,95.227787,...,0.0,8673.0,8673.0,0.0,0,0,2,939017.0,Creciente,1
2,9,2026-01-08 12:22:40,5974028.4,10,36,Independiente,4036000,829000,529554,47.613894,...,0.0,18702.0,18702.0,0.0,3,0,0,NaN,NaN,0
3,4,2025-08-04 12:04:10,1671240.0,6,48,Empleado,1524547,498000,252420,95.227787,...,0.0,15782.0,15782.0,0.0,3,0,0,1536193.0,Creciente,1
4,9,2025-04-26 11:24:26,2781636.0,11,44,Empleado,5000000,4000000,217037,95.227787,...,0.0,204804.0,204804.0,0.0,3,0,1,933473.0,Creciente,1


## 3. Verificar tipos, faltantes y duplicados

In [3]:
summary = pd.DataFrame({"tipo": df.dtypes.astype(str), "nulos": df.isna().sum(), "valores_distintos": df.nunique()})
display(summary)
print("Duplicados completos:", df.duplicated().sum())
assert df.shape == (10763, 23), "La fuente cambió: revisar el análisis."
assert set(df.Pago_atiempo.unique()) == {0, 1}

,tipo,nulos,valores_distintos
tipo_credito,int64,0,6
fecha_prestamo,datetime64[us],0,10758
capital_prestado,float64,0,7306
plazo_meses,int64,0,18
edad_cliente,int64,0,54
tipo_laboral,str,0,2
salario_cliente,int64,0,1385
total_otros_prestamos,int64,0,1538
cuota_pactada,int64,0,9836
puntaje,float64,0,248


Duplicados completos: 0


## 4. Comprobar el objetivo y las fechas

In [4]:
display(df.Pago_atiempo.value_counts().rename_axis("Pago_atiempo").to_frame("registros"))
print("Inicio:", df.fecha_prestamo.min())
print("Fin:", df.fecha_prestamo.max())

,registros
Pago_atiempo,
1,10252
0,511


Inicio: 2024-11-26 09:17:04
Fin: 2026-04-26 18:43:52


## 5. Exportar a CSV y comprobar la lectura

In [5]:
destination = ROOT / "Base_de_datos.csv"
df.to_csv(destination, index=False, encoding="utf-8")
reloaded = load_data(destination)
original_check = df.copy()
# CSV no conserva tipos mixtos dentro de una columna: comparar su representación.
for col in df.select_dtypes("object").columns:
    original_check[col] = original_check[col].astype("string")
    reloaded[col] = reloaded[col].astype("string")
pd.testing.assert_frame_equal(original_check, reloaded, check_exact=False, rtol=1e-10)
print("CSV exportado y comprobado:", destination.name)

CSV exportado y comprobado: Base_de_datos.csv


C:\Users\david\AppData\Local\Temp\ipykernel_45612\237109212.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes("object").columns:


## Resultado
Se cargaron 10.763 registros y 23 columnas, sin duplicados completos. La fuente contiene 511 valores `0` y 10.252 valores `1` en el objetivo. El CSV conserva los faltantes y los valores; las categorías numéricas mezcladas con texto se representan como texto al releer: la limpieza para el modelo se realiza dentro del pipeline.

## Próximo paso
Abrir `comprension_eda.ipynb`. La exploración que orienta decisiones del modelo utilizará únicamente el bloque de entrenamiento.